# Qdrant Ingestion

Embeds Bangladesh legal acts with a HuggingFace **sentence-transformers** model (run locally on CPU) and upserts them into Qdrant.

For a one-shot run use the script: `uv run python notebooks/ingest_qdrant.py`.

Config is read from the project `.env`: `QDRANT_VECTORESTORE`, `QDRANT_API_KEY`, `QDRANT_COLLECTION`, `EMBEDDING_MODEL`, `HF_TOKEN`.

In [ ]:
import hashlib
import json
import os
import re
import uuid
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.http import models
from sentence_transformers import SentenceTransformer

In [ ]:
load_dotenv(Path("../.env"))
load_dotenv()

QDRANT_URL = os.getenv("QDRANT_VECTORESTORE", "http://213.136.80.53:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY") or None
COLLECTION_NAME = os.getenv("QDRANT_COLLECTION", "legal_acts_event_rag_full")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
HF_TOKEN = os.getenv("HF_TOKEN") or None
ACTS_DIR = Path("../data/acts")

MAX_CHARS_PER_CHUNK = 1200
CHUNK_OVERLAP = 100
UPSERT_BATCH_SIZE = 64
ENCODE_BATCH_SIZE = 32

if not EMBEDDING_MODEL:
    raise ValueError("EMBEDDING_MODEL is required in .env")

# e5-family models are asymmetric: passages get a "passage:" prefix at ingestion,
# queries get a "query:" prefix at search time (handled in the API embedding module).
IS_E5 = "e5" in EMBEDDING_MODEL.lower()
EMBEDDING_MODEL

In [ ]:
def build_qdrant_client() -> QdrantClient:
    parsed = urlparse(QDRANT_URL)
    kwargs = {"url": QDRANT_URL, "api_key": QDRANT_API_KEY, "timeout": 120}
    # qdrant-client defaults to port 6333 when the URL omits a port; an https
    # endpoint behind a reverse proxy (e.g. Cloudflare) is served on 443.
    if parsed.scheme == "https" and parsed.port is None:
        kwargs["port"] = 443
    return QdrantClient(**kwargs)


qdrant_client = build_qdrant_client()
qdrant_client.get_collections()

In [ ]:
SECTION_INDEX_RE = re.compile(
    r"^[\s\"'\[\]]*\[?([0-9\u09E6-\u09EF]+[a-zA-Z]*)[.\u0964\u09F7\-\s]"
)
FOOTNOTE_MARKER_RE = re.compile(r"\d+\[(.*?)\]")
VOID_SECTION_RE = re.compile(
    r"\[\s*(Omitted|Repealed?|Rep\.)\s+by" r"|\[\s*Repeal\.\-" r"|\[\s*Omit\.\-",
    re.IGNORECASE,
)
SUBSECTION_SPLIT_RE = re.compile(r"(?=(?:\(\d+\)|\([a-zA-Z]+\)))")

In [ ]:
def extract_section_index(section_content: str) -> str:
    if not section_content:
        return "Unknown"
    match = SECTION_INDEX_RE.search(section_content)
    return (match.group(1).strip() if match else "Unknown") or "Unknown"

In [ ]:
def clean_section_content(section_content: str) -> str:
    if not section_content:
        return ""
    return FOOTNOTE_MARKER_RE.sub(r"\1", section_content).strip()

In [ ]:
def chunk_section_content(
    text: str, max_chars: int = 1200, overlap: int = 100
) -> list[str]:
    if not text:
        return []
    if len(text) <= max_chars:
        return [text]

    def slice_with_overlap(value: str) -> list[str]:
        chunks, start = [], 0
        while start < len(value):
            end = min(start + max_chars, len(value))
            chunks.append(value[start:end].strip())
            if end >= len(value):
                break
            start = max(0, end - overlap)
        return [chunk for chunk in chunks if chunk]

    # Prefer semantic-ish splits on subsection markers before fixed-size slicing.
    parts = [part.strip() for part in SUBSECTION_SPLIT_RE.split(text) if part.strip()]
    if len(parts) > 1:
        merged, current = [], ""
        for part in parts:
            candidate = f"{current} {part}".strip() if current else part
            if len(candidate) <= max_chars:
                current = candidate
                continue

            if current:
                merged.append(current)
                current = ""

            if len(part) <= max_chars:
                current = part
                continue

            merged.extend(slice_with_overlap(part))

        if current:
            merged.append(current)
        if merged:
            return merged

    return slice_with_overlap(text)

In [ ]:
def build_embedding_text(
    act_title: str, section_index: str, chunk_part: int, chunk_text: str
) -> str:
    return (
        f"Act: {act_title}\nSection {section_index} (Part {chunk_part}): {chunk_text}"
    )


def passage_text(text: str) -> str:
    return f"passage: {text}" if IS_E5 else text

In [ ]:
def collect_all_records(
    acts_dir: Path, max_chars: int = 1200, overlap: int = 100
) -> tuple[list[dict], dict]:
    records = []
    stats = {
        "files_seen": 0,
        "acts_skipped_repealed": 0,
        "acts_skipped_no_sections": 0,
        "sections_seen": 0,
        "sections_skipped_repealed": 0,
        "sections_skipped_empty": 0,
        "chunks_created": 0,
    }

    for file_path in sorted(acts_dir.glob("act-print-*.json")):
        stats["files_seen"] += 1
        with open(file_path, "r", encoding="utf-8") as f:
            act_obj = json.load(f)

        if act_obj.get("csv_metadata", {}).get("is_repealed") is True:
            stats["acts_skipped_repealed"] += 1
            continue

        sections = act_obj.get("sections") or []
        if not sections:
            stats["acts_skipped_no_sections"] += 1
            continue

        for section in sections:
            stats["sections_seen"] += 1
            raw_content = (section or {}).get("section_content", "")
            if VOID_SECTION_RE.search(raw_content or ""):
                stats["sections_skipped_repealed"] += 1
                continue

            cleaned = clean_section_content(raw_content)
            if not cleaned:
                stats["sections_skipped_empty"] += 1
                continue

            section_index = extract_section_index(cleaned)

            chunks = chunk_section_content(
                cleaned, max_chars=max_chars, overlap=overlap
            )
            for chunk_part, chunk_text in enumerate(chunks, start=1):
                payload = {
                    "act_title": act_obj.get("act_title"),
                    "act_no": act_obj.get("act_no"),
                    "act_year": (
                        int(act_obj["act_year"])
                        if str(act_obj.get("act_year", "")).isdigit()
                        else None
                    ),
                    "section_index": section_index,
                    "chunk_part": chunk_part,
                    "language": act_obj.get("language"),
                    "govt_system": act_obj.get("government_context", {}).get(
                        "govt_system"
                    ),
                    "source_url": act_obj.get("source_url"),
                    "section_content_clean": chunk_text,
                }
                records.append(
                    {
                        "embedding_text": build_embedding_text(
                            act_title=act_obj.get("act_title", "Unknown Act"),
                            section_index=section_index,
                            chunk_part=chunk_part,
                            chunk_text=chunk_text,
                        ),
                        "payload": payload,
                    }
                )

    stats["chunks_created"] = len(records)
    return records, stats

In [ ]:
records, stats = collect_all_records(
    ACTS_DIR, max_chars=MAX_CHARS_PER_CHUNK, overlap=CHUNK_OVERLAP
)

stats

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL, device="cpu", token=HF_TOKEN)
model

In [ ]:
def embed_passages(
    model: SentenceTransformer, texts: list[str], batch_size: int = ENCODE_BATCH_SIZE
) -> list[list[float]]:
    if not texts:
        return []
    prefixed = [passage_text(t) for t in texts]
    vectors = model.encode(
        prefixed,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return [vec.tolist() for vec in vectors]

In [ ]:
import numpy as np


def cosine_similarity(a, b):
    a = np.array(a).flatten()
    b = np.array(b).flatten()
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


texts_to_embed = [
    "খনি ও খনিজ সম্পদ (নিয়ন্ত্রণ ও উন্নয়ন) আইন, ১৯৯২",
    "Khoni o Khonij Shompod (Niyontron o Unnayan) Ain, 1992",
    "Mines and Mineral Resources (Control and Development) Act, 1992",
]
query = "Mines and Mineral Resources (Control and Development) Act, 1992"

doc_vecs = embed_passages(model, texts_to_embed)
# e5 query side uses the "query:" prefix
query_vec = model.encode(
    f"query: {query}" if IS_E5 else query, normalize_embeddings=True
)

for text, vec in sorted(
    zip(texts_to_embed, doc_vecs),
    key=lambda tv: cosine_similarity(query_vec, tv[1]),
    reverse=True,
):
    print(f"{text} -> {cosine_similarity(query_vec, vec):.4f}")

In [ ]:
def recreate_qdrant_collection(
    client: QdrantClient, collection_name: str, vector_size: int
) -> None:
    existing = {c.name for c in client.get_collections().collections}
    if collection_name in existing:
        client.delete_collection(collection_name=collection_name)

    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=vector_size, distance=models.Distance.COSINE
        ),
    )
    client.create_payload_index(
        collection_name=collection_name,
        field_name="act_year",
        field_schema=models.PayloadSchemaType.INTEGER,
        wait=True,
    )
    client.create_payload_index(
        collection_name=collection_name,
        field_name="language",
        field_schema=models.PayloadSchemaType.KEYWORD,
        wait=True,
    )

In [ ]:
def ingest_acts_to_qdrant(
    *,
    client: QdrantClient,
    model: SentenceTransformer,
    records: list[dict],
    collection_name: str,
    batch_size: int,
    max_records: int | None = None,
) -> dict:
    if max_records is not None:
        records = records[:max_records]

    if not records:
        print("[ingest] No records to ingest.")
        return {"points_indexed": 0, "vector_size": None}

    points_indexed = 0
    vector_size = None
    collection_ready = False
    total_batches = (len(records) + batch_size - 1) // batch_size

    for batch_num, start in enumerate(range(0, len(records), batch_size), start=1):
        batch = records[start : start + batch_size]
        vectors = embed_passages(model, [r["embedding_text"] for r in batch])

        if not collection_ready:
            vector_size = len(vectors[0])
            print(f"[ingest] Recreating '{collection_name}' (size={vector_size})")
            recreate_qdrant_collection(client, collection_name, vector_size)
            collection_ready = True

        points = [
            models.PointStruct(
                id=str(uuid.uuid4()), vector=vec, payload=rec["payload"]
            )
            for rec, vec in zip(batch, vectors)
        ]
        client.upsert(collection_name=collection_name, points=points, wait=True)
        points_indexed += len(points)
        print(f"[ingest] Batch {batch_num}/{total_batches} -> {points_indexed} points")

    return {"points_indexed": points_indexed, "vector_size": vector_size}

In [ ]:
ingestion_summary = ingest_acts_to_qdrant(
    client=qdrant_client,
    model=model,
    records=records,
    collection_name=COLLECTION_NAME,
    batch_size=UPSERT_BATCH_SIZE,
)

ingestion_summary

In [ ]:
qdrant_client.get_collection(COLLECTION_NAME).model_dump()